In [ ]:
# 1. Login (Do this once)
from huggingface_hub import login
from huggingface_hub import create_repo, upload_folder
login("#hf_secret")

In [2]:
MODEL_ID = "microsoft/Phi-4-mini-instruct"
OUTPUT_DIR_25 = "/kaggle/working/wanda/Phi-4-mini-instruct-WANDA-25"
REPO_ID_25 = f"EdgeCompress01/Phi-4-mini-instruct-WANDA-25"
REPO_ID = f"EdgeCompress01/Phi-4-mini-instruct-WANDA"

OUTPUT_DIR_50 = "/kaggle/working/wanda/Phi-4-mini-instruct-WANDA-50"
REPO_ID_50 = f"EdgeCompress01/Phi-4-mini-instruct-WANDA-50"

OUTPUT_DIR_70 = "/kaggle/working/wanda/Phi-4-mini-instruct-WANDA-70"
REPO_ID_70 = f"EdgeCompress01/Phi-4-mini-instruct-WANDA-70"
SEED = 42

# LLAMA

In [3]:
!pip install -q accelerate==1.0.1 datasets==3.0.0
!git clone https://github.com/locuslab/wanda.git
%cd wanda

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 330.9/330.9 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 474.3/474.3 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 14.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.31.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.6.1 which is incompatible.
Cloning into 'wanda'...
remote: Enumerating objects: 150, done.
remote: Counting objects: 100% (91/91), done.
remote: Compressing objects: 100% (28/28), done.
remote: Total 150 (delta 68), reused 63 (delta 63), pack-reused 59 (from 1)
Receiving objects: 100% (150/150), 119.77 KiB | 7.04 MiB/s, done.
Resolving deltas: 100% (74/74), done.
/kaggle/working/wanda


In [4]:
!pip install -q "transformers==4.49.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 86.2 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 51.5 MB/s eta 0:00:00a 0:00:01


In [5]:
import os

file_path = '/kaggle/working/wanda/lib/data.py'

with open(file_path, 'r') as f:
    content = f.read()

# Remove the legacy config name 'allenai--c4'
fixed_content = content.replace("'allenai/c4', 'allenai--c4'", "'allenai/c4'")

with open(file_path, 'w') as f:
    f.write(fixed_content)

print("Fixed lib/data.py successfully!")

Fixed lib/data.py successfully!


In [ ]:
# # Install/Update the required quantization libraries
# !pip install -U bitsandbytes accelerate

## Change the Main file to have the wanda only

In [6]:
%%writefile main.py
import argparse
import os 
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from lib.prune import prune_wanda, check_sparsity

def get_llm(model_id, cache_dir):
    model = AutoModelForCausalLM.from_pretrained(
        model_id, 
        torch_dtype=torch.float16, 
        low_cpu_mem_usage=True, 
        device_map="cpu",
        trust_remote_code=True # Phi models often need this
    )
    # Phi-4-mini has a large context, but we'll prune using 2048 for VRAM safety
    model.seqlen = 2048
    return model

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--model', type=str, default="microsoft/Phi-4-mini-instruct")
    parser.add_argument('--seed', type=int, default=0)
    parser.add_argument('--nsamples', type=int, default=32) 
    parser.add_argument('--sparsity_ratio', type=float, default=0.25)
    parser.add_argument("--cache_dir", default="llm_weights", type=str )
    parser.add_argument('--save_model', type=str, default="pruned_phi4_mini")
    
    args = parser.parse_args()

    model = get_llm(args.model, args.cache_dir)
    tokenizer = AutoTokenizer.from_pretrained(args.model, trust_remote_code=True)
    device = torch.device("cuda:0")
    
    print(f"Pruning Phi-4-mini: {args.model}")
    prune_wanda(args, model, tokenizer, device)

    print("*"*30)
    s_ratio = check_sparsity(model)
    print(f"Sparsity check: {s_ratio*100:.2f}%")
    print("*"*30)

    model.save_pretrained(args.save_model)
    tokenizer.save_pretrained(args.save_model)

if __name__ == '__main__':
    main()

Overwriting main.py


## Change the prune file

In [7]:
%%writefile lib/prune.py
import torch
import torch.nn as nn
import gc

def find_layers(module, layers=[nn.Linear], name=''):
    if type(module) in layers:
        return {name: module}
    res = {}
    for name1, child in module.named_children():
        res.update(find_layers(child, layers=layers, name=name + '.' + name1 if name != '' else name1))
    return res

def prepare_calibration_input(model, dataloader, device):
    layers = model.model.layers
    
    # Phi-4-mini specific bookends
    model.model.embed_tokens.to(device)
    if hasattr(model.model, 'rotary_emb'):
        model.model.rotary_emb.to(device)
    
    dtype = next(iter(model.parameters())).dtype
    inps = torch.zeros((len(dataloader), model.seqlen, model.config.hidden_size), dtype=dtype, device='cpu')
    cache = {'i': 0, 'attention_mask': None, 'position_ids': None}

    class Catcher(nn.Module):
        def __init__(self, module):
            super().__init__()
            self.module = module
        def forward(self, inp, **kwargs):
            inps[cache['i']] = inp.detach().cpu()
            cache['i'] += 1
            if kwargs.get('attention_mask') is not None:
                cache['attention_mask'] = kwargs['attention_mask'].detach().cpu()
            if kwargs.get('position_ids') is not None:
                cache['position_ids'] = kwargs['position_ids'].detach().cpu()
            raise ValueError
            
    layers[0] = Catcher(layers[0].to(device))
    for batch in dataloader:
        try:
            with torch.no_grad():
                model(batch[0].to(device))
        except ValueError:
            pass
            
    layers[0] = layers[0].module.cpu()
    model.model.embed_tokens.cpu()
    torch.cuda.empty_cache()
    
    outs = torch.zeros_like(inps)
    return inps, outs, cache['attention_mask'], cache['position_ids']

def prune_wanda(args, model, tokenizer, device, prune_n=0, prune_m=0):
    from .data import get_loaders
    print("Loading calibration data...")
    dataloader, _ = get_loaders("c4", nsamples=args.nsamples, seed=args.seed, seqlen=model.seqlen, tokenizer=tokenizer)
    
    inps, outs, attention_mask, position_ids = prepare_calibration_input(model, dataloader, device)
    
    if attention_mask is not None: attention_mask = attention_mask.to(device)
    if position_ids is not None: position_ids = position_ids.to(device)
    
    if hasattr(model.model, 'rotary_emb'):
        model.model.rotary_emb.to(device)

    layers = model.model.layers
    for i in range(len(layers)):
        print(f"Pruning Layer {i}...")
        layer = layers[i].to(device)
        subset = find_layers(layer)

        handles = []
        for name in subset:
            def get_hook(name):
                def hook(_, inp, out):
                    if not hasattr(subset[name], 'scaler_row'):
                        subset[name].scaler_row = torch.zeros(subset[name].weight.shape[1], device=device)
                    subset[name].scaler_row += torch.norm(inp[0].data, p=2, dim=[0, 1]).float()**2
                return hook
            handles.append(subset[name].register_forward_hook(get_hook(name)))
        
        # Consistent forward pass for Phi-4-mini
        for j in range(args.nsamples):
            with torch.no_grad():
                curr_in = inps[j].unsqueeze(0).to(device)
                # Phi models compute cos/sin based on the whole model's rotary_emb
                cos, sin = model.model.rotary_emb(curr_in, position_ids)
                layer(curr_in, attention_mask=attention_mask, position_ids=position_ids, position_embeddings=(cos, sin))
            
        for h in handles: h.remove()

        for name in subset:
            W = subset[name].weight.data
            scaling_factor = torch.sqrt(subset[name].scaler_row).to(W.dtype).reshape((1, -1))
            W_metric = torch.abs(W) * scaling_factor
            thresh = torch.sort(W_metric.flatten())[0][int(W_metric.numel() * args.sparsity_ratio)]
            W[W_metric <= thresh] = 0
            del subset[name].scaler_row
            torch.cuda.empty_cache()

        # Second pass to get inputs for the next layer
        for j in range(args.nsamples):
            with torch.no_grad():
                curr_in = inps[j].unsqueeze(0).to(device)
                cos, sin = model.model.rotary_emb(curr_in, position_ids)
                layer_out = layer(curr_in, attention_mask=attention_mask, position_ids=position_ids, position_embeddings=(cos, sin))[0]
                outs[j] = layer_out.detach().cpu()
        
        inps.data = outs.data 
        layers[i] = layer.cpu()
        del layer
        gc.collect()
        torch.cuda.empty_cache()

    print("Pruning finished.")

def check_sparsity(model):
    all_weights = 0
    zero_weights = 0
    for name, p in model.named_parameters():
        if 'weight' in name:
            all_weights += p.numel()
            zero_weights += (p == 0).sum().item()
    return float(zero_weights) / all_weights if all_weights > 0 else 0.0

Overwriting lib/prune.py


In [8]:
!touch lib/__init__.py

## Wanda 25

In [11]:
import torch
import gc
gc.collect()
torch.cuda.empty_cache()
# This helps prevent the fragmentation mentioned in your error log
%env PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True


import time

start = time.time()

!python main.py \
    --model {MODEL_ID} \
    --nsamples 64 \
    --sparsity_ratio 0.25 \
    --seed {SEED} \
    --save_model {OUTPUT_DIR_25}

end = time.time()

elapsed = end - start

hours = int(elapsed // 3600)
minutes = int((elapsed % 3600) // 60)
seconds = elapsed % 60
milliseconds = elapsed * 1000

print(f"Time taken: {hours}h {minutes}m {seconds:.2f}s")
print(f"Milliseconds: {milliseconds:.2f} ms")


env: PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
config.json: 2.50kB [00:00, 1.66MB/s]
configuration_phi3.py: 10.9kB [00:00, 10.5MB/s]
A new version of the following files was downloaded from https://huggingface.co/microsoft/Phi-4-mini-instruct:
- configuration_phi3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
modeling_phi3.py: 54.3kB [00:00, 129MB/s]
A new version of the following files was downloaded from https://huggingface.co/microsoft/Phi-4-mini-instruct:
- modeling_phi3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
2026-05-02 20:46:06.996210: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777754767.186375     156 cuda_

In [12]:
import torch
from transformers import AutoModelForCausalLM

def check_pruned_model(model_path):
    model = AutoModelForCausalLM.from_pretrained(model_path, torch_dtype="auto", device_map="cpu")
    
    total_params = sum(p.numel() for p in model.parameters())
    # count_nonzero is the key for unstructured pruning
    active_params = sum(torch.count_nonzero(p).item() for p in model.parameters())
    
    sparsity = 100 * (1 - active_params / total_params)
    print(f"Model: {model_path}")
    print(f"Total Params: {total_params / 1e9:.2f}B")
    print(f"Active Params: {active_params / 1e9:.2f}B")
    print(f"Effective Sparsity: {sparsity:.2f}%")

# Usage
check_pruned_model(OUTPUT_DIR_25)

2026-05-02 20:50:21.138871: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777755021.230960      75 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777755021.239227      75 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777755021.266670      75 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777755021.266692      75 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777755021.266695      75 computation_placer.cc:177] computation placer alr

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Model: /kaggle/working/wanda/Phi-4-mini-instruct-WANDA-25
Total Params: 3.84B
Active Params: 3.03B
Effective Sparsity: 21.00%


In [13]:
import torch
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(OUTPUT_DIR_25, torch_dtype=torch.float16, device_map="cpu")

# Check a specific layer (e.g., Layer 5 Down Proj)
weight = model.model.layers[5].mlp.down_proj.weight.data
zeros = (weight == 0).sum().item()
total = weight.numel()

print(f"Layer 5 Down Proj Sparsity: {zeros/total*100:.2f}%")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Layer 5 Down Proj Sparsity: 25.01%


In [14]:
# create repo in organization
create_repo(REPO_ID, repo_type="model", exist_ok=True)

# upload to organization repo
upload_folder(
    repo_id=REPO_ID,
    repo_type="model",
    folder_path=OUTPUT_DIR_25,
    path_in_repo="Models/25",
    commit_message="Upload WANAD Pruningmodel"
)

print("Upload complete to organization!")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Upload complete to organization!


In [ ]:
!rm -rf /kaggle/working/wanda/Llama-3.2-3B-Instruct-WANDA-25

## Wanda 50

In [15]:
import torch
import gc
gc.collect()
torch.cuda.empty_cache()
# This helps prevent the fragmentation mentioned in your error log
%env PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True


import time

start = time.time()

!python main.py \
    --model {MODEL_ID} \
    --nsamples 64 \
    --sparsity_ratio 0.5 \
    --seed {SEED} \
    --save_model {OUTPUT_DIR_50}

end = time.time()

elapsed = end - start

hours = int(elapsed // 3600)
minutes = int((elapsed % 3600) // 60)
seconds = elapsed % 60
milliseconds = elapsed * 1000

print(f"Time taken: {hours}h {minutes}m {seconds:.2f}s")
print(f"Milliseconds: {milliseconds:.2f} ms")


env: PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
2026-05-02 20:52:20.960994: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777755140.983503     370 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777755140.990035     370 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777755141.008413     370 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777755141.008441     370 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777755141.008445    

In [16]:
import torch
from transformers import AutoModelForCausalLM

def check_pruned_model(model_path):
    model = AutoModelForCausalLM.from_pretrained(model_path, torch_dtype="auto", device_map="cpu")
    
    total_params = sum(p.numel() for p in model.parameters())
    # count_nonzero is the key for unstructured pruning
    active_params = sum(torch.count_nonzero(p).item() for p in model.parameters())
    
    sparsity = 100 * (1 - active_params / total_params)
    print(f"Model: {model_path}")
    print(f"Total Params: {total_params / 1e9:.2f}B")
    print(f"Active Params: {active_params / 1e9:.2f}B")
    print(f"Effective Sparsity: {sparsity:.2f}%")

# Usage
check_pruned_model(OUTPUT_DIR_50)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Model: /kaggle/working/wanda/Phi-4-mini-instruct-WANDA-50
Total Params: 3.84B
Active Params: 2.22B
Effective Sparsity: 42.00%


In [17]:
import torch
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(OUTPUT_DIR_50, torch_dtype=torch.float16, device_map="cpu")

# Check a specific layer (e.g., Layer 5 Down Proj)
weight = model.model.layers[5].mlp.down_proj.weight.data
zeros = (weight == 0).sum().item()
total = weight.numel()

print(f"Layer 5 Down Proj Sparsity: {zeros/total*100:.2f}%")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Layer 5 Down Proj Sparsity: 50.03%


In [18]:
# create repo in organization
create_repo(REPO_ID, repo_type="model", exist_ok=True)

# upload to organization repo
upload_folder(
    repo_id=REPO_ID,
    repo_type="model",
    folder_path=OUTPUT_DIR_50,
    path_in_repo="Models/50",
    commit_message="Upload WANAD Pruningmodel"
)

print("Upload complete to organization!")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Upload complete to organization!


In [ ]:
!rm -rf /kaggle/working/wanda/Llama-3.2-3B-Instruct-WANDA-50

## Wanda 70

In [9]:
import torch
import gc
gc.collect()
torch.cuda.empty_cache()
# This helps prevent the fragmentation mentioned in your error log
%env PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True


import time

start = time.time()

!python main.py \
    --model {MODEL_ID} \
    --nsamples 64 \
    --sparsity_ratio 0.7 \
    --seed {SEED} \
    --save_model {OUTPUT_DIR_70}

end = time.time()

elapsed = end - start

hours = int(elapsed // 3600)
minutes = int((elapsed % 3600) // 60)
seconds = elapsed % 60
milliseconds = elapsed * 1000

print(f"Time taken: {hours}h {minutes}m {seconds:.2f}s")
print(f"Milliseconds: {milliseconds:.2f} ms")


env: PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
config.json: 2.50kB [00:00, 1.47MB/s]
configuration_phi3.py: 10.9kB [00:00, 7.61MB/s]
A new version of the following files was downloaded from https://huggingface.co/microsoft/Phi-4-mini-instruct:
- configuration_phi3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
modeling_phi3.py: 54.3kB [00:00, 118MB/s]
A new version of the following files was downloaded from https://huggingface.co/microsoft/Phi-4-mini-instruct:
- modeling_phi3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
2026-05-02 20:59:15.209125: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777755555.462072     111 cuda_

In [10]:
import torch
from transformers import AutoModelForCausalLM

def check_pruned_model(model_path):
    model = AutoModelForCausalLM.from_pretrained(model_path, torch_dtype="auto", device_map="cpu")
    
    total_params = sum(p.numel() for p in model.parameters())
    # count_nonzero is the key for unstructured pruning
    active_params = sum(torch.count_nonzero(p).item() for p in model.parameters())
    
    sparsity = 100 * (1 - active_params / total_params)
    print(f"Model: {model_path}")
    print(f"Total Params: {total_params / 1e9:.2f}B")
    print(f"Active Params: {active_params / 1e9:.2f}B")
    print(f"Effective Sparsity: {sparsity:.2f}%")

# Usage
check_pruned_model(OUTPUT_DIR_70)

2026-05-02 21:03:26.076682: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777755806.171573      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777755806.180078      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777755806.213511      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777755806.213532      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777755806.213535      57 computation_placer.cc:177] computation placer alr

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Model: /kaggle/working/wanda/Phi-4-mini-instruct-WANDA-70
Total Params: 3.84B
Active Params: 1.58B
Effective Sparsity: 58.79%


In [11]:
import torch
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(OUTPUT_DIR_70, torch_dtype=torch.float16, device_map="cpu")

# Check a specific layer (e.g., Layer 5 Down Proj)
weight = model.model.layers[5].mlp.down_proj.weight.data
zeros = (weight == 0).sum().item()
total = weight.numel()

print(f"Layer 5 Down Proj Sparsity: {zeros/total*100:.2f}%")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Layer 5 Down Proj Sparsity: 70.02%


In [12]:
# create repo in organization
create_repo(REPO_ID, repo_type="model", exist_ok=True)

# upload to organization repo
upload_folder(
    repo_id=REPO_ID,
    repo_type="model",
    folder_path=OUTPUT_DIR_70,
    path_in_repo="Models/70",
    commit_message="Upload WANAD Pruningmodel"
)

print("Upload complete to organization!")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Upload complete to organization!


In [ ]:
!rm -rf /kaggle/working/wanda/Llama-3.2-3B-Instruct-WANDA-70

## Test the model

In [ ]:
import torch
from transformers import AutoTokenizer

# 1. Setup
model_path = "/kaggle/working/wanda/pruned_llama_3_2"
tokenizer = AutoTokenizer.from_pretrained(model_path)
prompt = "The best way to prune a neural network is"

# 2. Ensure model is on GPU
model.to("cuda")

# 3. Manually tokenize and move inputs to CUDA
# This ensures the 'index' (input_ids) is on the same device as the weights
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

# 4. Generate using the model directly (more reliable than pipeline here)
with torch.no_grad():
    output_ids = model.generate(
        **inputs, 
        max_new_tokens=50, 
        do_sample=True, 
        temperature=0.7,
        pad_token_id=tokenizer.eos_token_id
    )

# 5. Decode and Print
response = tokenizer.decode(output_ids[0], skip_special_tokens=True)
print(response)

In [ ]:
# Step 1: The Perplexity Script

import torch
from tqdm import tqdm
from datasets import load_dataset

def evaluate_perplexity(model, tokenizer, dataset_name="wikitext", dataset_config="wikitext-2-raw-v1"):
    # 1. Load the test set
    test_data = load_dataset(dataset_name, dataset_config, split="test")
    encodings = tokenizer("\n\n".join(test_data["text"]), return_tensors="pt")

    max_length = 2048 # Llama 3.2 context window
    stride = 512
    seq_len = encodings.input_ids.size(1)

    nlls = []
    prev_end_loc = 0
    
    print(f"Calculating Perplexity for {seq_len} tokens...")
    
    for begin_loc in tqdm(range(0, seq_len, stride)):
        end_loc = min(begin_loc + max_length, seq_len)
        trg_len = end_loc - prev_end_loc  # may be different from stride on last loop
        input_ids = encodings.input_ids[:, begin_loc:end_loc].to("cuda")
        target_ids = input_ids.clone()
        target_ids[:, :-trg_len] = -100

        with torch.no_grad():
            outputs = model(input_ids, labels=target_ids)
            # loss is calculated using CrossEntropyLoss which averages over valid labels
            neg_log_likelihood = outputs.loss

        nlls.append(neg_log_likelihood * trg_len)

        prev_end_loc = end_loc
        if end_loc == seq_len:
            break

    ppl = torch.exp(torch.stack(nlls).sum() / end_loc)
    return ppl.item()

# Run the evaluation
# ppl_value = evaluate_perplexity(model, tokenizer)
# print(f"\nFinal Perplexity: {ppl_value:.4f}")

In [ ]:
import time

def mobile_sim_benchmark(model, tokenizer, prompt="Explain AI in 10 words."):
    # 1. Force CPU and limit threads to simulate mobile processor
    torch.set_num_threads(4) 
    model.to("cpu")
    
    inputs = tokenizer(prompt, return_tensors="pt")
    
    # Warm start
    _ = model.generate(**inputs, max_new_tokens=1)
    
    # Start Timing
    start = time.time()
    outputs = model.generate(**inputs, max_new_tokens=30, do_sample=True)
    end = time.time()
    
    # Calculate Metrics
    tokens_gen = len(outputs[0]) - len(inputs[0])
    tps = tokens_gen / (end - start)
    
    print(f"--- Mobile Simulation Result ---")
    print(f"Tokens/Sec (TPS): {tps:.2f}")
    print(f"Latency: {end - start:.2f}s")
    
    if tps < 5:
        print("Status: Too slow for phone (Needs 4-bit Quantization)")
    else:
        print("Status: Good for Edge!")

# mobile_sim_benchmark(model, tokenizer)

In [ ]:
# !python main.py \
#     --model "meta-llama/Llama-3.2-3B-Instruct" \
#     --prune_method wanda \
#     --sparsity_ratio 0.25 \
#     --sparsity_type unstructured \
#     --save out/llama_3.2_pruned/

# !python main.py \
#     --model meta-llama/Llama-3.2-3B-Instruct \
#     --prune_method wanda \
#     --sparsity_ratio 0.5 \
#     --sparsity_type unstructured \
#     --save_model out/llama_3.2_pruned/

# Important Kaggle Tip: Since you have cuda:0 detected now, the process will be much faster. 
# However, Llama 3.2 3B uses a lot of memory during the "Wanda" phase (where it calculates the $X^TX$ Hessian matrix). 
# If you hit an Out of Memory (OOM) error, reduce the --nsamples from the default 128 to 64.

# !python main.py \
#     --model meta-llama/Llama-3.2-3B-Instruct \
#     --prune_method wanda \
#     --sparsity_ratio 0.25 \
#     --sparsity_type unstructured \
#     --nsamples 32 \
#     --save out/llama_3_2_pruned/